# 09 · Análisis causal: ¿reaccionar con un cambio tras un gol en contra aumenta la probabilidad de anotar?

**Bloque agregado para el criterio de identificación causal (25%).** El notebook 06 midió que, tras una sustitución, las métricas del equipo cambian (posesión, altura, presión) — pero un "antes vs. después" **no es causalidad**: el equipo podría cambiar por el simple paso del tiempo, el cansancio del rival, o porque se juega de forma distinta al ir perdiendo. Aquí se hace la pregunta con un diseño que sí distingue correlación de efecto ajustado por confounders.

**Pregunta causal:** cuando América recibe un gol que lo deja perdiendo o empatando (antes del minuto 75), ¿hacer una sustitución en los 10 minutos siguientes **causa** un aumento en la probabilidad de anotar en los 15 minutos posteriores al gol recibido?

- **Tratamiento:** al menos una sustitución propia en (t_gol, t_gol + 10 min].
- **Resultado:** al menos un gol propio en (t_gol, t_gol + 15 min].
- **Unidad de análisis:** cada gol en contra elegible (no el partido completo).
- **Confounders emparejados:** situación tras el gol (perdiendo vs. empatando) y minuto en que ocurrió (bloques de 15 min). Un gol que deja perdiendo al minuto 10 no es comparable con uno que deja empatando al minuto 70.
- **Método:** matching exacto por estratos (*coarsened exact matching*), con intervalo de confianza por bootstrap y análisis de sensibilidad. Se eligió por ser interpretable sin conocimientos de ML — cada comparación se puede leer como una tabla.

In [ ]:
import sys
sys.path.append("../src")
import pandas as pd
import numpy as np

from framework_analitico import dataset_causal_reaccion_gol_contra, matching_estratificado

EQUIPO = "América"  # cambia esto para reaplicar el analisis a otro entrenador/equipo

# Solo se cargan las columnas necesarias (el CSV completo pesa ~530 MB y tarda >2 min en leerse)
columnas = ["match_id", "period", "minute", "second", "type", "team", "shot_outcome", "player", "substitution_replacement"]
events = pd.read_csv("../data/raw/events_jardine_america.csv", usecols=columnas, low_memory=False)
partidos = pd.read_csv("../data/raw/partidos_jardine_america.csv")
print(f"Eventos cargados: {len(events)}")

## 1. Dataset de eventos elegibles

Cada fila es un gol en contra que dejó a América perdiendo o empatando antes del minuto 75.

In [ ]:
dataset = dataset_causal_reaccion_gol_contra(events, equipo=EQUIPO, partidos_df=partidos)

print(f"Eventos elegibles: {len(dataset)}")
print()
print("Situación tras el gol:")
print(dataset["situacion"].value_counts())
print()
print("Minuto del gol (bloques de 15 min):")
print(dataset["minuto_bucket"].value_counts().sort_index())
print()
print("Tratamiento (1 = hubo cambio en los 10 min siguientes):")
print(dataset["tratamiento"].value_counts())

## 2. Comparación ingenua (solo correlación)

Compara tratados vs. controles **en bruto**, sin tomar en cuenta que no son comparables entre sí (por ejemplo, casi no hay cambios en los primeros 15 minutos, y los goles tempranos pesan en el grupo de control).

In [ ]:
ingenua = dataset.groupby("tratamiento")["resultado"].agg(["mean", "sum", "count"]).round(3)
ingenua.columns = ["tasa_anota_en_15min", "partidos_que_anotaron", "n"]
ingenua.index = ["Sin cambio (control)", "Con cambio (tratamiento)"]
ingenua

## 3. Matching estratificado

Se compara tratados vs. controles **solo dentro de cada estrato** (situación × bloque de minuto), donde sí son comparables, y se promedian las diferencias ponderando por el tamaño del estrato. Los estratos sin tratados o sin controles no se pueden comparar y se excluyen (se listan explícitamente).

In [ ]:
resultado = matching_estratificado(dataset, "tratamiento", "resultado", ["situacion", "minuto_bucket"])

print(f"Diferencia ingenua (sin emparejar): {resultado['diferencia_ingenua']:+.3f}")
print(f"  tasa tratados: {resultado['tasa_tratados_bruta']:.3f} | tasa controles: {resultado['tasa_controles_bruta']:.3f}")
print(f"Efecto emparejado por estratos:     {resultado['efecto_estimado']:+.3f}")
print(f"  IC 95% (bootstrap, 2000 remuestreos): {resultado['ic95_bootstrap']}")
print(f"  n tratados: {resultado['n_tratados_total']} | n controles: {resultado['n_controles_total']}")
print()
print("Comparación dentro de cada estrato:")
resultado["tabla_estratos"]

In [ ]:
print("Estratos excluidos (sin tratados o sin controles, no hay con qué comparar):")
pd.DataFrame(resultado["estratos_sin_variacion"])

## 4. Sensibilidad: ¿la conclusión depende de las ventanas elegidas?

Las ventanas de reacción (10 min) y de resultado (15 min) son decisiones del análisis, no hechos. Si la conclusión cambiara mucho al moverlas, no sería confiable.

In [ ]:
filas = []
for ventana_reaccion, ventana_resultado in [(5, 15), (10, 10), (10, 15), (10, 20), (15, 15)]:
    ds = dataset_causal_reaccion_gol_contra(
        events, equipo=EQUIPO, partidos_df=partidos,
        ventana_reaccion_min=ventana_reaccion, ventana_resultado_min=ventana_resultado,
    )
    r = matching_estratificado(ds, "tratamiento", "resultado", ["situacion", "minuto_bucket"])
    filas.append({
        "ventana_reaccion_min": ventana_reaccion,
        "ventana_resultado_min": ventana_resultado,
        "n_tratados": r["n_tratados_total"],
        "diferencia_ingenua": r["diferencia_ingenua"],
        "efecto_emparejado": r["efecto_estimado"],
        "ic95_bootstrap": r["ic95_bootstrap"],
    })
pd.DataFrame(filas)

## 5. Síntesis y limitaciones

**No hay evidencia de que reaccionar con un cambio tras un gol en contra aumente la probabilidad de anotar en los 15 minutos siguientes — ni de que la reduzca.** De los 88 goles en contra elegibles, América anotó en los 15 minutos posteriores en 11.1% de los casos con cambio (3 de 27) y en 19.7% sin cambio (12 de 61): una diferencia ingenua de −8.6 puntos porcentuales. Leída en bruto, sugeriría que cambiar "perjudica". Al emparejar por situación y minuto, el efecto baja a −5.8 pp, con un intervalo de confianza al 95% de [−24.2, +11.6] pp: incluye con holgura el cero, así que los datos son compatibles con un efecto nulo, con uno moderadamente negativo y con uno moderadamente positivo.

**La conclusión es estable ante las decisiones de diseño.** Al variar las ventanas (reacción de 10 y 15 min; resultado de 10, 15 y 20 min) el efecto emparejado se mantiene entre −3 y −8 pp y todos los intervalos incluyen el cero. La ventana de reacción de 5 minutos se descarta: solo 8 eventos quedan como tratados, el estimador se vuelve inestable (intervalo de más de 80 pp de ancho: [−32.5, +51.1]) y el signo incluso se invierte (+25.5 pp), otra señal de que con tan pocos eventos el ruido domina.

**El efecto emparejado es frágil: depende de unos pocos eventos.** Dentro de los estratos el signo no es consistente: perdiendo en el minuto 15 y en el 30 la diferencia es negativa (−14.3 y −42.9 pp), perdiendo en el 45 es positiva (+13.1 pp) y el resto es cero. Un solo estrato (perdiendo, minuto 30; 12 de los 65 eventos comparables) concentra buena parte del −5.8 pp: ahí anotaron 3 de 7 controles y 0 de 5 tratados. Además, el emparejamiento descarta 23 eventos (todos controles sin tratados comparables), por lo que la estimación se apoya en 65 eventos, no en 88.

**Qué cambia entre correlación y efecto ajustado.** La diferencia ingenua (−8.6 pp) y la emparejada (−5.8 pp) no son iguales: parte de lo que parecía un efecto negativo se explica por *cuándo* y *en qué situación* ocurren los cambios (los goles tempranos casi no tienen cambio posterior; los goles que dejan perdiendo pesan distinto que los que empatan). Pero ni siquiera el efecto ajustado es distinguible de cero — por lo que no se puede afirmar causalidad en ningún sentido.

**Lectura operativa.** Un cambio inmediato tras un gol en contra no es, por sí solo, una palanca demostrada para anotar pronto. Esto es coherente con el notebook 06 (los cambios sí modifican la posesión, la altura y la presión del equipo), pero esos cambios de proceso no se traducen de forma detectable en goles en una ventana corta: con estos datos el valor de una sustitución tendría que buscarse en otras métricas, no en la probabilidad de gol a 15 minutos.

**Limitaciones explícitas (no resueltas):**
- **Poca potencia estadística:** con 88 eventos (27 tratados), solo se detectarían efectos grandes. "No hay evidencia de efecto" no equivale a "no hay efecto".
- **El tratamiento no es aleatorio y mezcla causas:** muchos cambios son planificados (alrededor del minuto 60-70) y habrían ocurrido igual sin el gol; eso contamina el grupo tratado y sesga el efecto hacia cero. No se separó cambio reactivo de cambio planificado.
- **Confounders no controlados:** fuerza del rival, tipo de cambio (ofensivo o no) y jugador que entra, tarjetas o expulsiones, y estado físico. El supuesto de identificación es que, dentro de cada estrato (situación × minuto), el momento del cambio es "como aleatorio" — un supuesto fuerte y no verificable con estos datos.
- **Independencia:** un mismo partido puede aportar más de un evento; se tratan como independientes (sin ajuste por conglomerado), lo que subestima la incertidumbre real.
- **Estratos sin variación:** los goles de los primeros 15 minutos no tienen ningún cambio posterior, así que esa zona del partido queda fuera de la comparación.

**Contribución al reto:** el diseño separa explícitamente correlación (diferencia ingenua) de efecto ajustado por confounders (matching), cuantifica la incertidumbre (bootstrap) y prueba su robustez (sensibilidad). Es reutilizable: `matching_estratificado` no depende de fútbol ni de América, y sirve para cualquier pregunta causal con tratamiento y resultado binarios.